# LLM Data Quality Assessment

## Import Libraries

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.llm_client import call_llm, LLM_PROVIDER
from utils.loader import load_tables
import json
import matplotlib.patches as mpatches

BASE_DQA_INPUT_PATH = RECONCILED_DATA_DIR
BASE_LLM_OUTPUT_PATH = DQA_REPORT_DIR / "LLM"
BASE_LLM_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

dfs = load_tables("reconciled", normalize_cols="lower")


def is_llm_error(text):
    return isinstance(text, str) and text.startswith("[LLM Error]")


print("Libraries imported from utils. LLM client ready.")
print(f"LLM provider: {LLM_PROVIDER.upper()}")

## LLM DQA Pipeline

In [ ]:
for table_name, df in dfs.items():
    if df is None or len(df) == 0:
        print(f"[{table_name}] SKIPPED: empty dataframe.")
        continue

    scorecard_path = DQA_REPORT_DIR / f"Scorecard_{table_name}.csv"

    # Check if both data and scorecard exist before processing
    if not scorecard_path.exists():
        print(f"[{table_name}] SKIPPED: Scorecard not found in {DQA_REPORT_DIR}.")
        continue

    print(f"\n{'='*60}\nSTARTING LLM PROCESSING FOR: {table_name.upper()}\n{'='*60}")

    # 1. Load data and scorecard
    df = df.copy()
    df.columns = df.columns.str.lower()
    dq_df = pd.read_csv(scorecard_path)

    # Building the Scorecard dictionary for the LLM context
    scorecard_context = {
        "table": table_name,
        "rows": len(df),
        "columns": list(df.columns),
        "overall_score": round(dq_df["Score"].mean(), 4),
        "dimensions": {row["Dimension"]: {"score": row["Score"], "issues": int(row["Issues"]), "details": row["Details"]} for _, row in dq_df.iterrows()},
    }

    # ---------------------------------------------------------
    # PHASE A: Automated Audit Report
    # ---------------------------------------------------------
    print(f"[{table_name}] 1/4 Generating Audit Report...")
    SYSTEM_AUDIT = """You are a Lead Data Quality Engineer in the aviation analytics sector.
    Analyze the provided Data Quality Assessment scorecard. Produce a structured, actionable audit report. 
    Prioritize issues based on their impact on data pipelines and analytical models. Use precise terminology."""

    PROMPT_AUDIT = f"""Below is the Data Quality Assessment scorecard for the '{table_name}' table.
    Produce a structured audit report containing exactly:
    1. Executive Summary (2-3 sentences regarding overall health).
    2. Critical Issues (Requires immediate remediation).
    3. Moderate Issues (Monitor or clean passively).
    4. Recommended Remediation Plan.
    5. Estimated Risk to Downstream Analytics.

    SCORECARD:
    {json.dumps(scorecard_context, indent=2)}"""

    audit_report = call_llm(PROMPT_AUDIT, system=SYSTEM_AUDIT)
    if is_llm_error(audit_report):
        print(f"[{table_name}] SKIPPED: audit generation failed -> {audit_report}")
        continue
    with open(BASE_LLM_OUTPUT_PATH / f"Audit_{table_name}.md", "w", encoding="utf-8") as f:
        f.write(audit_report)

    # ---------------------------------------------------------
    # PHASE B: Generating Pandas Validation Rules
    # ---------------------------------------------------------
    print(f"[{table_name}] 2/4 Generating Pandas Validation Rules...")
    sample_info = {}
    for col in df.columns:
        non_null = df[col].dropna()
        sample_info[col] = {"dtype": str(df[col].dtype), "sample": non_null.head(3).tolist(), "n_null": int(df[col].isna().sum())}  # Only 3 to save tokens

    SYSTEM_RULES = """You are a Senior Python Data Engineer specializing in data pipelines.
    Generate Python pandas validation rule functions (lambdas) based on column metadata.
    Return ONLY valid Python code containing a dictionary named 'validity_rules'. No markdown outside comments."""

    PROMPT_RULES = f"""Generate Python pandas validation rules for the {table_name} dataset.
    Output must be exactly a Python dictionary named 'validity_rules' (keys=columns, values=lambdas returning boolean Series).
    Column metadata: {json.dumps(sample_info, default=str)}"""

    generated_rules = call_llm(PROMPT_RULES, system=SYSTEM_RULES)
    if is_llm_error(generated_rules):
        print(f"[{table_name}] SKIPPED: rules generation failed -> {generated_rules}")
        continue
    with open(BASE_LLM_OUTPUT_PATH / f"Rules_{table_name}.py", "w", encoding="utf-8") as f:
        f.write(generated_rules.replace("```python", "").replace("```", "").strip())

    # ---------------------------------------------------------
    # PHASE C: Stakeholder Explanation
    # ---------------------------------------------------------
    print(f"[{table_name}] 3/4 Generating Stakeholder Explanation...")
    scores_text = "\n".join([f"- {dim}: {info['score']:.4f}" for dim, info in scorecard_context["dimensions"].items()])
    overall_text = f"- Overall: {scorecard_context['overall_score']:.4f}"

    SYSTEM_EXPLAIN = """You are a Data Governance Consultant presenting to a Chief Operating Officer (COO).
    Translate technical data quality scores into plain English. Focus on business risks. Avoid technical jargon."""

    PROMPT_EXPLAIN = f"""Our '{table_name}' table received these DQ scores:
    {scores_text}
    {overall_text}
    Explain what each score means for business operations. Conclude with a recommendation on data reliability."""

    explanation = call_llm(PROMPT_EXPLAIN, system=SYSTEM_EXPLAIN)
    if is_llm_error(explanation):
        print(f"[{table_name}] SKIPPED: stakeholder explanation failed -> {explanation}")
        continue
    with open(BASE_LLM_OUTPUT_PATH / f"Stakeholder_{table_name}.md", "w", encoding="utf-8") as f:
        f.write(explanation)

    # ---------------------------------------------------------
    # PHASE D: Priority Extraction & Plotting
    # ---------------------------------------------------------
    print(f"[{table_name}] 4/4 Generating Priority Weights and Plotting...")
    SYSTEM_PRIORITY = """You are a strictly deterministic LLM parser. 
    Convert text into numeric priority weights (0.0 to 1.0, where 1.0 is highest urgency).
    Return ONLY a valid Python dictionary named 'llm_priority'."""

    PROMPT_PRIORITY = f"""Based on this audit report: {audit_report[:1000]}...
    Assign urgency weights (0.0 to 1.0) to these dimensions: Completeness, Uniqueness, Validity, Consistency, Timeliness.
    Return EXACTLY this format:
    llm_priority = {{'Completeness': 0.X, 'Uniqueness': 0.X, 'Validity': 0.X, 'Consistency': 0.X, 'Timeliness': 0.X}}"""

    llm_priority_text = call_llm(PROMPT_PRIORITY, system=SYSTEM_PRIORITY)
    if is_llm_error(llm_priority_text):
        print(f"[{table_name}] SKIPPED: priority generation failed -> {llm_priority_text}")
        continue

    local_vars = {}
    try:
        clean_code = llm_priority_text.replace("```python", "").replace("```", "").strip()
        exec(clean_code, {}, local_vars)
        llm_priority = local_vars.get("llm_priority", {})
    except Exception as e:
        print(f"[{table_name}] Fallback parsing priorità: {e}")
        llm_priority = {dim: 0.5 for dim in scorecard_context["dimensions"].keys()}

    # Plotting (Classical vs LLM Priority)
    classical_scores = {row["Dimension"]: row["Score"] for _, row in dq_df.iterrows()}
    dims = list(classical_scores.keys())
    priority_values = [llm_priority.get(d, 0) for d in dims]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

    classical_vals = [classical_scores[d] for d in dims]

    ax1.bar(dims, classical_vals, color="#5dade2", alpha=0.85, width=0.5)

    ax1.set_ylim(0, 1.3)
    ax1.set_xlim(-0.5, len(dims))

    ax1.set_title(f"Classical DQA Scores: {table_name.upper()}", fontsize=14, fontweight="bold", pad=25, loc="left", color="#333333")

    ax1.axhline(0.95, color="#cccccc", linestyle="--", linewidth=1.2, zorder=0)
    testo_bg = dict(facecolor="white", edgecolor="none", pad=3, alpha=1.0)
    ax1.text(len(dims) - 0.4, 0.95, "Target (95%)", color="#888888", fontsize=10, ha="left", va="center", bbox=testo_bg, zorder=2)

    for i, v in enumerate(classical_vals):
        ax1.text(i, v + 0.03, f"{v:.2%}", ha="center", va="bottom", fontsize=11, fontweight="semibold", color="#333333")

    for spine in ["top", "right", "left"]:
        ax1.spines[spine].set_visible(False)
    ax1.spines["bottom"].set_color("#dddddd")
    ax1.set_yticks([])
    ax1.tick_params(axis="x", length=0, labelsize=11, labelcolor="#555555")

    colors = ["#e74c3c" if p >= 0.8 else ("#f1c40f" if p >= 0.6 else "#2ecc71") for p in priority_values]

    ax2.bar(dims, priority_values, color=colors, alpha=0.85, width=0.5)

    ax2.set_ylim(0, 1.3)
    ax2.set_title(f"LLM Remediation Priority: {table_name.upper()}", fontsize=14, fontweight="bold", pad=25, loc="left", color="#333333")

    for i, v in enumerate(priority_values):
        ax2.text(i, v + 0.03, f"{v:.2%}", ha="center", va="bottom", fontsize=11, fontweight="semibold", color="#333333")

    legend_elements = [
        mpatches.Patch(color="#e74c3c", label="High (≥80%)", alpha=0.85),
        mpatches.Patch(color="#f1c40f", label="Medium (≥60%)", alpha=0.85),
        mpatches.Patch(color="#2ecc71", label="Low (<60%)", alpha=0.85),
    ]
    ax2.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False, fontsize=10, labelcolor="#555555")

    for spine in ["top", "right", "left"]:
        ax2.spines[spine].set_visible(False)
    ax2.spines["bottom"].set_color("#dddddd")
    ax2.set_yticks([])
    ax2.tick_params(axis="x", length=0, labelsize=11, labelcolor="#555555")

    plt.subplots_adjust(wspace=0.15)
    plt.tight_layout()
    plt.savefig(BASE_LLM_OUTPUT_PATH / f"PriorityPlot_{table_name}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("\nAll tables processed. LLM outputs saved in:", BASE_LLM_OUTPUT_PATH)